# 02 Feature Engineering And Validation

Leakage-safe feature generation and contamination-aware folds.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from data_loading import discover_data_dir
from teknofest.data_prep import prepare_data
from teknofest.features import FeatureEngineer, detect_binary_al_cols
from teknofest.validation import contamination_aware_folds, fold_summary

prepared = prepare_data(discover_data_dir(PROJECT_ROOT))
flag_cols = detect_binary_al_cols(prepared.master, prepared.al_cols)
engineer = FeatureEngineer(prepared.al_cols, prepared.al_raw, flag_cols)
engineered = engineer.fit_transform(prepared.master)
feature_cols = [c for c in engineered.columns if c not in {"Variant_ID", "Label"}]
len(feature_cols), feature_cols[:20]

In [ ]:
folds = contamination_aware_folds(prepared.master["Label"], prepared.master_shared_mask)
diagnostics = fold_summary(prepared.master, folds)
diagnostics

In [ ]:
ax = diagnostics.set_index("fold")[["train_size", "val_size"]].plot(kind="bar", color=["#52796f", "#b56576"])
ax.set_title("Fold sizes")
ax.set_ylabel("Rows")
plt.tight_layout()